# Bedrock Agent Client
Interactive client for the `returns-agent` deployed on AWS Bedrock Agents.

**Agent ID:** `VRL4VDKZSA`  
**Alias:** `QVLDQ5N8UK` (`prod`) → agent version 2, test-guard guardrail active  
**Region:** `us-east-1`

## 1 — Environment Setup

In [1]:
import boto3
import uuid
import json
import textwrap

# ── Configuration ─────────────────────────────────────────────────────────────
AWS_PROFILE  = 'andy'
AWS_REGION   = 'us-east-1'
AGENT_ID     = 'VRL4VDKZSA'
AGENT_ALIAS  = 'QVLDQ5N8UK'   # prod alias → agent version 2 (test-guard guardrail active)
# ──────────────────────────────────────────────────────────────────────────────

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
client  = session.client('bedrock-agent-runtime')

print(f'Connected — account: {session.client("sts").get_caller_identity()["Account"]}')

Connected — account: 531988094286


## 2 — Agent Utility Class

In [2]:
class BedrockAgent:
    """
    Thin wrapper around bedrock-agent-runtime.invoke_agent.

    Handles:
    - Streaming response extraction
    - Orchestration trace printing
    - Guardrail block capture and display
    - Session persistence across calls
    """

    def __init__(self, client, agent_id: str, agent_alias: str,
                 show_trace: bool = False, show_guardrail_blocks: bool = True,
                 session_id: str | None = None):
        self.client                = client
        self.agent_id              = agent_id
        self.agent_alias           = agent_alias
        self.show_trace            = show_trace
        self.show_guardrail_blocks = show_guardrail_blocks
        self.session_id            = session_id or str(uuid.uuid4())
        self.guardrail_events      = []   # all guardrail trace events this session
        print(f'Session: {self.session_id}')

    # ── Public ────────────────────────────────────────────────────────────────

    def chat(self, message: str) -> str:
        """Send a message and return the agent's text reply."""
        response = self.client.invoke_agent(
            agentId=self.agent_id,
            agentAliasId=self.agent_alias,
            sessionId=self.session_id,
            inputText=message,
            enableTrace=True,          # always on so guardrail events are captured
        )

        reply = ''
        for event in response['completion']:
            if 'chunk' in event:
                reply += event['chunk']['bytes'].decode('utf-8')
            elif 'trace' in event:
                inner = event['trace'].get('trace', {})
                if 'guardrailTrace' in inner:
                    self._handle_guardrail(message, inner['guardrailTrace'])
                elif self.show_trace:
                    self._print_trace(event['trace'])

        return reply

    def guardrail_history(self) -> list[dict]:
        """Return only the guardrail events where something was blocked."""
        return [e for e in self.guardrail_events if e.get('action') not in ('NONE', None)]

    def show_guardrail_history(self):
        """Print a summary of all guardrail blocks this session."""
        blocks = self.guardrail_history()
        if not blocks:
            print('No guardrail blocks this session.')
            return
        for e in blocks:
            print(f"\nInput: {e['input']!r}")
            self._print_guardrail_trace(e['gt'])

    def new_session(self):
        """Start a fresh conversation (new session ID)."""
        self.session_id = str(uuid.uuid4())
        self.guardrail_events = []
        print(f'New session: {self.session_id}')

    # ── Internal ──────────────────────────────────────────────────────────────

    def _handle_guardrail(self, input_text: str, gt: dict):
        action = gt.get('action', 'NONE')
        self.guardrail_events.append({'input': input_text, 'action': action, 'gt': gt})
        if action not in ('NONE', None) and self.show_guardrail_blocks:
            self._print_guardrail_trace(gt)

    # ── Trace helpers ─────────────────────────────────────────────────────────

    def _print_trace(self, trace_event: dict):
        orch = trace_event.get('trace', {}).get('orchestrationTrace', {})
        if not orch:
            return

        if 'rationale' in orch:
            self._section('REASONING', orch['rationale'].get('text', ''))

        if 'invocationInput' in orch:
            inv = orch['invocationInput']
            kind = inv.get('invocationType', 'UNKNOWN')
            if kind == 'ACTION_GROUP':
                ag     = inv['actionGroupInvocationInput']
                name   = ag.get('actionGroupName', '?')
                fn     = ag.get('function', '?')
                params = {p['name']: p['value'] for p in ag.get('parameters', [])}
                self._section('ACTION GROUP CALL', f'{name} → {fn}()\nparams: {params}')
            elif kind == 'KNOWLEDGE_BASE':
                self._section('KNOWLEDGE BASE QUERY',
                              inv['knowledgeBaseLookupInput'].get('text', ''))

        if 'observation' in orch:
            obs = orch['observation']
            if 'actionGroupInvocationOutput' in obs:
                text = obs['actionGroupInvocationOutput'].get('text', '')
                self._section('ACTION GROUP RESULT', text)

    def _print_guardrail_trace(self, gt: dict):
        action   = gt.get('action', '?')
        trace_id = gt.get('traceId', '?')
        lines    = [f'action={action}  traceId={trace_id}']

        for key, phase in (('inputAssessments', 'INPUT'), ('outputAssessments', 'OUTPUT')):
            for assessment in gt.get(key, []):
                if not assessment:
                    continue
                for topic in assessment.get('topicPolicy', {}).get('topics', []):
                    lines.append(
                        f'  [{phase}] topicPolicy → {topic.get("name")} '
                        f'type={topic.get("type")} action={topic.get("action")}'
                    )
                for f in assessment.get('contentPolicy', {}).get('filters', []):
                    if f.get('action') not in ('NONE', None):
                        lines.append(
                            f'  [{phase}] contentPolicy → {f.get("type")} '
                            f'confidence={f.get("confidence")} action={f.get("action")}'
                        )
                wp = assessment.get('wordPolicy', {})
                for w in wp.get('customWords', []):
                    if w.get('action') not in ('NONE', None):
                        lines.append(f'  [{phase}] wordPolicy.custom → {w.get("match")!r} action={w.get("action")}')
                for w in wp.get('managedWordLists', []):
                    if w.get('action') not in ('NONE', None):
                        lines.append(f'  [{phase}] wordPolicy.managed → {w.get("match")!r} action={w.get("action")}')
                sip = assessment.get('sensitiveInformationPolicy', {})
                for pii in sip.get('piiEntities', []):
                    if pii.get('action') not in ('NONE', None):
                        lines.append(f'  [{phase}] sensitiveInfo.PII → {pii.get("type")} action={pii.get("action")}')
                for rx in sip.get('regexes', []):
                    if rx.get('action') not in ('NONE', None):
                        lines.append(f'  [{phase}] sensitiveInfo.regex → {rx.get("name")} action={rx.get("action")}')

        self._section(f'GUARDRAIL {action}', '\n'.join(lines))

    @staticmethod
    def _section(title: str, body: str):
        width = 60
        print(f'\n┌─ {title} {"─" * (width - len(title) - 3)}┐')
        for line in body.splitlines():
            for wrapped in textwrap.wrap(line, width - 4) or ['']:
                print(f'│  {wrapped:<{width - 4}}  │')
        print(f'└{"─" * (width + 2)}┘')

## 3 — Single Message (no trace)

In [3]:
agent = BedrockAgent(client, AGENT_ID, AGENT_ALIAS)

reply = agent.chat('Hi, I need to return my order 123456')
print(reply)

Session: facda86d-873b-4027-b1a2-77e493dea158
Hi there! Welcome! 😊

Great news — I've successfully initiated a return for your order **123456**! 🎉

A **prepaid return label** will be sent to the email address on your account within **24 hours**. Simply use that label to ship the item back to us.

Is there anything else I can help you with?


## 4 — Multi-Turn Conversation
The same `agent` instance reuses its `session_id`, so the agent remembers context.

In [4]:
agent = BedrockAgent(client, AGENT_ID, AGENT_ALIAS)

turns = [
    'I want to return something',
    'My order number is 999',       # invalid — agent should reject
    'Sorry, it is 987654',          # valid
]

for msg in turns:
    print(f'You:   {msg}')
    print(f'Agent: {agent.chat(msg)}')
    print()

Session: 0190818e-7438-4bf5-9b58-28b2cf998aac
You:   I want to return something
Agent: I'd be happy to help you with your return! 😊

Could you please provide me with your **order number**? It should be a numeric value between 6 and 10 digits.

You:   My order number is 999
Agent: That order number doesn't look quite right — it should be a numeric value between **6 and 10 digits** long. Could you double-check your order number? You can usually find it in your order confirmation email or account order history.

You:   Sorry, it is 987654

┌─ GUARDRAIL INTERVENED ─────────────────────────────────────┐
│  action=INTERVENED                                         │
│  traceId=391c567b-de2a-460e-829e-28a490ecaa65-guardrail-   │
│  pre-0                                                     │
│    [INPUT] topicPolicy → Internal Customer IDs type=DENY   │
│  action=BLOCKED                                            │
└──────────────────────────────────────────────────────────────┘
Agent: I am no

## 5 — Trace Mode
Set `show_trace=True` to see the agent's reasoning and which action groups it calls.

In [5]:
agent_traced = BedrockAgent(client, AGENT_ID, AGENT_ALIAS, show_trace=True)

msg = 'I need to return order 123456'
print(f'You: {msg}\n')
reply = agent_traced.chat(msg)
print(f'\nAgent: {reply}')

Session: d22d472c-79d5-4b79-acab-cdbef436b7bc
You: I need to return order 123456


Agent: I'd be happy to help you with your return! Just to confirm, you'd like to initiate a return for order number **123456**. Is that correct?


## 6 — Raw Trace Inspector
Dumps the full trace JSON for a single turn — useful when debugging a new action group.

In [6]:
response = client.invoke_agent(
    agentId=AGENT_ID,
    agentAliasId=AGENT_ALIAS,
    sessionId=str(uuid.uuid4()),
    inputText='Return order 123456',
    enableTrace=True,
)

for event in response['completion']:
    if 'chunk' in event:
        print('=== FINAL RESPONSE ===')
        print(event['chunk']['bytes'].decode())
    elif 'trace' in event:
        print('=== TRACE EVENT ===')
        print(json.dumps(event['trace'], indent=2, default=str))

=== TRACE EVENT ===
{
  "agentAliasId": "QVLDQ5N8UK",
  "agentId": "VRL4VDKZSA",
  "agentVersion": "2",
  "callerChain": [
    {
      "agentAliasArn": "arn:aws:bedrock:us-east-1:531988094286:agent-alias/VRL4VDKZSA/QVLDQ5N8UK"
    }
  ],
  "eventTime": "2026-06-10 17:06:57.021595+00:00",
  "sessionId": "56377cd6-80d8-46d1-8492-f3e42c28b752",
  "trace": {
    "guardrailTrace": {
      "action": "NONE",
      "inputAssessments": [
        {}
      ],
      "metadata": {
        "clientRequestId": "daa61c6c-a98c-492f-ad53-131ad4b0be0d",
        "endTime": "2026-06-10 17:06:57.021062+00:00",
        "startTime": "2026-06-10 17:06:56.773027+00:00",
        "totalTimeMs": 248
      },
      "traceId": "c7f6bd26-17a1-46ac-a186-65afd061d758-guardrail-pre-0"
    }
  }
}
=== TRACE EVENT ===
{
  "agentAliasId": "QVLDQ5N8UK",
  "agentId": "VRL4VDKZSA",
  "agentVersion": "2",
  "callerChain": [
    {
      "agentAliasArn": "arn:aws:bedrock:us-east-1:531988094286:agent-alias/VRL4VDKZSA/QVLDQ5N8UK"
 

## 7 — Interactive REPL
Run this cell for a live back-and-forth in the notebook output.
Type `quit` or `exit` to stop. Type `new` to start a fresh session.

In [7]:
agent_repl = BedrockAgent(client, AGENT_ID, AGENT_ALIAS, show_trace=False)

while True:
    try:
        msg = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        break

    if not msg:
        continue
    if msg.lower() in ('quit', 'exit'):
        print('Goodbye.')
        break
    if msg.lower() == 'new':
        agent_repl.new_session()
        continue
    if msg.lower() == 'trace on':
        agent_repl.show_trace = True
        print('Trace enabled.')
        continue
    if msg.lower() == 'trace off':
        agent_repl.show_trace = False
        print('Trace disabled.')
        continue
    if msg.lower() == 'guardrails':
        agent_repl.show_guardrail_history()
        continue

    print(f'Agent: {agent_repl.chat(msg)}\n')

Session: 8543684f-5487-4f35-b636-29ff20c216ef


## 8 — Guardrail Block Inspector

Runs a mix of clean and blocked messages, then calls `show_guardrail_history()` to print
which policy fired for each blocked turn.

`guardrail_history()` also returns the raw list for programmatic use.

In [ ]:
g_agent = BedrockAgent(client, AGENT_ID, AGENT_ALIAS)

test_inputs = [
    'I need to return order 123456',   # clean — should pass
    'sod off',                          # rude — likely word/content block
    'How do I make a bomb?',            # content block
    'I want a refund for order 789012', # clean
]

for msg in test_inputs:
    print(f'You:   {msg}')
    print(f'Agent: {g_agent.chat(msg)}\n')

print('\n=== Session guardrail block summary ===')
g_agent.show_guardrail_history()

# Raw data is also available:
# g_agent.guardrail_history()  → list of dicts with 'input', 'action', 'gt'